# No-Feedback Domain-Wall Campaign Analysis

This notebook loads the completed no-feedback campaigns, validates per-sample outputs, generates observable plots, writes summary tables, and emits an executive summary. It does not rerun simulations.

In [ ]:

from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except Exception:
    display = None
    Markdown = None

ROOT = Path.cwd()
if ROOT.name != "class_A_fermionic_adaptive_circuit":
    # nbconvert may run from the notebook directory depending on invocation.
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "REPO_POLICY.md").exists() and (parent / "src").exists():
            ROOT = parent
            break

EXP_ROOT = ROOT / "experiments" / "no_feedback_domain_wall_chern"
RESULTS_ROOT = EXP_ROOT / "results"
OUT_DIR = EXP_ROOT / "analysis_outputs"
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"
for path in (OUT_DIR, FIG_DIR, TABLE_DIR):
    path.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "sans-serif",
    "font.sans-serif": ["CMU Sans Serif", "DejaVu Sans", "Arial"],
    "mathtext.fontset": "cm",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

CAMPAIGN_ORDER = [
    "uniform_alpha1_no_dw",
    "domain_wall_untruncated_alpha1_alpha30",
    "domain_wall_truncated_alpha1_alpha30",
]
INIT_ORDER = ["random", "maxmix"]
CAMPAIGN_LABELS = {
    "uniform_alpha1_no_dw": "Uniform alpha=1",
    "domain_wall_untruncated_alpha1_alpha30": "DW alpha=1/30, dwtrunc=0",
    "domain_wall_truncated_alpha1_alpha30": "DW alpha=1/30, dwtrunc=1",
}
INIT_LABELS = {"random": "Random pure", "maxmix": "Maxmix"}
COLORS = {"random": "#2563eb", "maxmix": "#c2410c"}


## Load And Validate Outputs

In [ ]:

def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as fh:
        return json.load(fh)


def load_npz(path: Path) -> dict[str, np.ndarray]:
    with np.load(path, allow_pickle=False) as payload:
        return {key: payload[key] for key in payload.files}

runs: list[dict[str, Any]] = []
validation_errors: list[str] = []

for campaign in CAMPAIGN_ORDER:
    for init_mode in INIT_ORDER:
        run_dir = RESULTS_ROOT / campaign / init_mode
        cfg_path = run_dir / "run_config.json"
        if not cfg_path.exists():
            validation_errors.append(f"Missing run_config.json: {cfg_path}")
            continue
        cfg = load_json(cfg_path)
        samples = int(cfg.get("samples", -1))
        cycles = int(cfg.get("cycles", -1))
        sample_paths = sorted((run_dir / "samples").glob("sample_*.npz"))
        if len(sample_paths) != samples:
            validation_errors.append(f"{run_dir}: expected {samples} sample files, found {len(sample_paths)}")
        chern_vs_cycle = pd.read_csv(run_dir / "chern_vs_cycle.csv")
        chern_summary = pd.read_csv(run_dir / "chern_summary.csv")
        local_summary = load_npz(run_dir / "local_chern_marker_summary.npz")
        transfer_summary = None
        if campaign != "uniform_alpha1_no_dw":
            transfer_path = run_dir / "transfer_spectrum_summary.npz"
            if not transfer_path.exists():
                validation_errors.append(f"Missing transfer_spectrum_summary.npz: {transfer_path}")
            else:
                transfer_summary = load_npz(transfer_path)
        sample_payloads = []
        for sample_path in sample_paths:
            payload = load_npz(sample_path)
            required = {"cycle", "real_space_chern", "final_local_chern_marker", "final_G", "sample_seed", "sample_index"}
            missing = sorted(required - set(payload))
            if missing:
                validation_errors.append(f"{sample_path}: missing {missing}")
            if payload.get("cycle", np.array([])).shape != (cycles + 1,):
                validation_errors.append(f"{sample_path}: bad cycle shape {payload.get('cycle', np.array([])).shape}")
            if payload.get("real_space_chern", np.array([])).shape != (cycles + 1,):
                validation_errors.append(f"{sample_path}: bad real_space_chern shape {payload.get('real_space_chern', np.array([])).shape}")
            if payload.get("final_local_chern_marker", np.array([])).shape != (int(cfg["Nx"]), int(cfg["Ny"])):
                validation_errors.append(f"{sample_path}: bad marker shape {payload.get('final_local_chern_marker', np.array([])).shape}")
            if campaign != "uniform_alpha1_no_dw":
                transfer_required = {"transfer_singular_values", "transfer_exponents", "transfer_gap", "finite_eigenstate_count", "choi_failure_records_json"}
                missing_transfer = sorted(transfer_required - set(payload))
                if missing_transfer:
                    validation_errors.append(f"{sample_path}: missing transfer arrays {missing_transfer}")
            sample_payloads.append(payload)
        runs.append({
            "campaign": campaign,
            "init_mode": init_mode,
            "run_dir": run_dir,
            "config": cfg,
            "chern_vs_cycle": chern_vs_cycle,
            "chern_summary": chern_summary,
            "local_summary": local_summary,
            "transfer_summary": transfer_summary,
            "sample_payloads": sample_payloads,
        })

expected_runs = len(CAMPAIGN_ORDER) * len(INIT_ORDER)
expected_samples = expected_runs * 10
actual_samples = sum(len(run["sample_payloads"]) for run in runs)
if len(runs) != expected_runs:
    validation_errors.append(f"Expected {expected_runs} runs, found {len(runs)}")
if actual_samples != expected_samples:
    validation_errors.append(f"Expected {expected_samples} total sample files, found {actual_samples}")

if validation_errors:
    print("Validation errors:")
    for error in validation_errors:
        print(" -", error)
    raise RuntimeError("Output validation failed")

print(f"Validated {len(runs)} runs and {actual_samples} per-sample files.")
print(f"Results root: {RESULTS_ROOT}")
print(f"Analysis output: {OUT_DIR}")


## Summary Metrics

In [ ]:

def sem(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=np.float64)
    if values.size <= 1:
        return 0.0
    return float(np.std(values, ddof=1) / math.sqrt(values.size))

run_rows = []
final_rows = []
local_rows = []
transfer_rows = []

for run in runs:
    campaign = run["campaign"]
    init_mode = run["init_mode"]
    cfg = run["config"]
    sample_payloads = run["sample_payloads"]
    final_cherns = np.asarray([payload["real_space_chern"][-1] for payload in sample_payloads], dtype=np.float64)
    initial_cherns = np.asarray([payload["real_space_chern"][0] for payload in sample_payloads], dtype=np.float64)
    local_stack = np.stack([payload["final_local_chern_marker"] for payload in sample_payloads], axis=0)
    full_marker_mean_per_sample = np.mean(local_stack, axis=(1, 2))
    row_base = {
        "campaign": campaign,
        "campaign_label": CAMPAIGN_LABELS[campaign],
        "init_mode": init_mode,
        "init_label": INIT_LABELS[init_mode],
        "samples": len(sample_payloads),
        "cycles": int(cfg["cycles"]),
        "Nx": int(cfg["Nx"]),
        "Ny": int(cfg["Ny"]),
        "radius": float(cfg["radius"]),
        "nshell": cfg["nshell"],
        "p_gain_effective": float(cfg["p_gain_effective"]),
        "p_loss_effective": float(cfg["p_loss_effective"]),
    }
    run_rows.append(row_base)
    final_rows.append({
        **row_base,
        "initial_chern_mean": float(np.mean(initial_cherns)),
        "initial_chern_std": float(np.std(initial_cherns, ddof=1)) if initial_cherns.size > 1 else 0.0,
        "initial_chern_sem": sem(initial_cherns),
        "final_chern_mean": float(np.mean(final_cherns)),
        "final_chern_std": float(np.std(final_cherns, ddof=1)) if final_cherns.size > 1 else 0.0,
        "final_chern_sem": sem(final_cherns),
        "delta_chern_mean": float(np.mean(final_cherns - initial_cherns)),
    })
    local_row = {
        **row_base,
        "local_marker_full_mean": float(np.mean(full_marker_mean_per_sample)),
        "local_marker_full_std": float(np.std(full_marker_mean_per_sample, ddof=1)) if full_marker_mean_per_sample.size > 1 else 0.0,
        "local_marker_full_sem": sem(full_marker_mean_per_sample),
    }
    if "domain_wall_geometry_diagnostics" in cfg and cfg["domain_wall_geometry_diagnostics"].get("checked"):
        x0, x1 = cfg["domain_wall_geometry_diagnostics"]["DW_loc"]
        slab_vals = np.mean(local_stack[:, x0:x1 + 1, :], axis=(1, 2))
        outside_parts = []
        if x0 > 0:
            outside_parts.append(local_stack[:, :x0, :].reshape(local_stack.shape[0], -1))
        if x1 + 1 < local_stack.shape[1]:
            outside_parts.append(local_stack[:, x1 + 1:, :].reshape(local_stack.shape[0], -1))
        outside_vals = np.mean(np.concatenate(outside_parts, axis=1), axis=1) if outside_parts else np.full(local_stack.shape[0], np.nan)
        local_row.update({
            "dw_x_min": int(x0),
            "dw_x_max": int(x1),
            "local_marker_slab_mean": float(np.mean(slab_vals)),
            "local_marker_slab_std": float(np.std(slab_vals, ddof=1)) if slab_vals.size > 1 else 0.0,
            "local_marker_slab_sem": sem(slab_vals),
            "local_marker_outside_mean": float(np.nanmean(outside_vals)),
        })
    local_rows.append(local_row)

    if run["transfer_summary"] is not None:
        ts = run["transfer_summary"]
        gaps = np.asarray(ts["transfer_gap"], dtype=np.float64)
        finite_counts = np.asarray(ts["finite_eigenstate_count"], dtype=np.int64)
        choi_active = np.asarray(ts["choi_active_final"], dtype=bool)
        transfer_rows.append({
            **row_base,
            "transfer_gap_mean": float(np.mean(gaps)),
            "transfer_gap_std": float(np.std(gaps, ddof=1)) if gaps.size > 1 else 0.0,
            "transfer_gap_sem": sem(gaps),
            "transfer_gap_min": float(np.min(gaps)),
            "transfer_gap_max": float(np.max(gaps)),
            "finite_eigenstate_count_min": int(np.min(finite_counts)),
            "finite_eigenstate_count_max": int(np.max(finite_counts)),
            "choi_active_all": bool(np.all(choi_active)),
            "choi_inactive_count": int(np.count_nonzero(~choi_active)),
            "choi_basis_size": int(np.asarray(ts["choi_basis_size"])[0]),
        })

run_summary = pd.DataFrame(run_rows)
final_chern_summary = pd.DataFrame(final_rows)
local_marker_region_summary = pd.DataFrame(local_rows)
transfer_gap_summary = pd.DataFrame(transfer_rows)

run_summary.to_csv(TABLE_DIR / "run_summary.csv", index=False)
final_chern_summary.to_csv(TABLE_DIR / "final_chern_summary.csv", index=False)
local_marker_region_summary.to_csv(TABLE_DIR / "local_marker_region_summary.csv", index=False)
transfer_gap_summary.to_csv(TABLE_DIR / "transfer_gap_summary.csv", index=False)

print("Final Chern summary:")
display(final_chern_summary[["campaign_label", "init_label", "final_chern_mean", "final_chern_sem", "delta_chern_mean"]])
print("Transfer gap summary:")
display(transfer_gap_summary[["campaign_label", "init_label", "transfer_gap_mean", "transfer_gap_sem", "choi_active_all", "choi_basis_size"]])


## Plot Helpers

In [ ]:

def savefig(fig: mpl.figure.Figure, stem: str) -> None:
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{stem}.{ext}", bbox_inches="tight", dpi=300)


def add_dw_lines(ax, cfg: dict[str, Any]) -> None:
    geom = cfg.get("domain_wall_geometry_diagnostics", {})
    if geom.get("checked"):
        x0, x1 = geom["DW_loc"]
        for x in (x0 - 0.5, x1 + 0.5):
            ax.axhline(x, color="white", lw=0.8, ls="--", alpha=0.9)
            ax.axhline(x, color="black", lw=0.35, ls="--", alpha=0.7)


def run_for(campaign: str, init_mode: str) -> dict[str, Any]:
    for run in runs:
        if run["campaign"] == campaign and run["init_mode"] == init_mode:
            return run
    raise KeyError((campaign, init_mode))


## Real-Space Chern Number Vs Cycle

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(10.125, 2.6), sharey=True, constrained_layout=True)
for ax, campaign in zip(axes, CAMPAIGN_ORDER):
    for init_mode in INIT_ORDER:
        run = run_for(campaign, init_mode)
        df = run["chern_summary"]
        raw = run["chern_vs_cycle"]
        for _, sdf in raw.groupby("sample_index"):
            ax.plot(sdf["cycle"], sdf["real_space_chern"], color=COLORS[init_mode], alpha=0.12, lw=0.7)
        ax.plot(df["cycle"], df["real_space_chern_mean"], color=COLORS[init_mode], lw=1.8, label=INIT_LABELS[init_mode])
        ax.fill_between(
            df["cycle"].to_numpy(),
            (df["real_space_chern_mean"] - df["real_space_chern_sem"]).to_numpy(),
            (df["real_space_chern_mean"] + df["real_space_chern_sem"]).to_numpy(),
            color=COLORS[init_mode], alpha=0.18, lw=0,
        )
    ax.set_title(CAMPAIGN_LABELS[campaign], fontsize=8)
    ax.set_xlabel("Cycle")
    ax.grid(True, alpha=0.25, lw=0.5)
axes[0].set_ylabel("Real-space Chern")
axes[-1].legend(loc="best", fontsize=7)
savefig(fig, "chern_vs_cycle_by_campaign")
plt.show()

fig, ax = plt.subplots(figsize=(3.375, 2.6), constrained_layout=True)
linestyles = {
    "uniform_alpha1_no_dw": "-",
    "domain_wall_untruncated_alpha1_alpha30": "--",
    "domain_wall_truncated_alpha1_alpha30": ":",
}
for campaign in CAMPAIGN_ORDER:
    for init_mode in INIT_ORDER:
        run = run_for(campaign, init_mode)
        df = run["chern_summary"]
        ax.plot(
            df["cycle"], df["real_space_chern_mean"],
            color=COLORS[init_mode], ls=linestyles[campaign], lw=1.5,
            label=f"{CAMPAIGN_LABELS[campaign]} / {INIT_LABELS[init_mode]}",
        )
ax.set_xlabel("Cycle")
ax.set_ylabel("Real-space Chern")
ax.grid(True, alpha=0.25, lw=0.5)
ax.legend(fontsize=5.5, ncol=1)
savefig(fig, "chern_vs_cycle_overview")
plt.show()


## Steady-State Local Chern Marker

In [ ]:

all_maps = []
for campaign in CAMPAIGN_ORDER:
    for init_mode in INIT_ORDER:
        marker = run_for(campaign, init_mode)["local_summary"]["final_local_chern_marker_mean"]
        all_maps.append(marker)
vlim = float(np.nanmax(np.abs(np.stack(all_maps))))
vlim = max(vlim, 1e-6)

fig, axes = plt.subplots(3, 2, figsize=(6.75, 8.0), constrained_layout=True)
for r, campaign in enumerate(CAMPAIGN_ORDER):
    for c, init_mode in enumerate(INIT_ORDER):
        ax = axes[r, c]
        run = run_for(campaign, init_mode)
        marker = run["local_summary"]["final_local_chern_marker_mean"]
        im = ax.imshow(marker, origin="lower", cmap="RdBu_r", vmin=-vlim, vmax=vlim, aspect="equal")
        add_dw_lines(ax, run["config"])
        ax.set_title(f"{CAMPAIGN_LABELS[campaign]}\n{INIT_LABELS[init_mode]}", fontsize=8)
        ax.set_xlabel("y")
        ax.set_ylabel("x")
fig.colorbar(im, ax=axes, shrink=0.72, label="Mean final local marker")
savefig(fig, "local_chern_marker_grid")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(10.125, 2.9), constrained_layout=True)
diff_maps = []
for campaign in CAMPAIGN_ORDER:
    maxmix = run_for(campaign, "maxmix")["local_summary"]["final_local_chern_marker_mean"]
    random = run_for(campaign, "random")["local_summary"]["final_local_chern_marker_mean"]
    diff_maps.append(maxmix - random)
dvlim = max(float(np.nanmax(np.abs(np.stack(diff_maps)))), 1e-6)
for ax, campaign, diff in zip(axes, CAMPAIGN_ORDER, diff_maps):
    im = ax.imshow(diff, origin="lower", cmap="PuOr_r", vmin=-dvlim, vmax=dvlim, aspect="equal")
    add_dw_lines(ax, run_for(campaign, "maxmix")["config"])
    ax.set_title(CAMPAIGN_LABELS[campaign], fontsize=8)
    ax.set_xlabel("y")
    ax.set_ylabel("x")
fig.colorbar(im, ax=axes, shrink=0.82, label="Maxmix - random")
savefig(fig, "local_chern_marker_init_differences")
plt.show()


## Transfer Spectra And Gaps

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(6.75, 2.7), constrained_layout=True)
for ax, campaign in zip(axes, CAMPAIGN_ORDER[1:]):
    for init_mode in INIT_ORDER:
        ts = run_for(campaign, init_mode)["transfer_summary"]
        exponents = np.asarray(ts["transfer_exponents"], dtype=np.float64)
        finite = np.where(np.isfinite(exponents), exponents, np.nan)
        mean = np.nanmean(finite, axis=0)
        # Plot sorted by absolute exponent to emphasize near-gap region.
        order = np.argsort(np.abs(mean))
        near = mean[order][:80]
        ax.plot(np.arange(near.size), near, color=COLORS[init_mode], lw=1.5, label=INIT_LABELS[init_mode])
        for row in finite:
            row_near = row[order][:80]
            ax.plot(np.arange(row_near.size), row_near, color=COLORS[init_mode], alpha=0.10, lw=0.6)
    ax.axhline(0, color="black", lw=0.6, alpha=0.5)
    ax.set_title(CAMPAIGN_LABELS[campaign], fontsize=8)
    ax.set_xlabel("Near-gap mode rank")
    ax.set_ylabel("Transfer exponent")
    ax.grid(True, alpha=0.25, lw=0.5)
axes[-1].legend(fontsize=7)
savefig(fig, "transfer_exponent_spectra")
plt.show()

fig, ax = plt.subplots(figsize=(3.375, 2.7), constrained_layout=True)
positions = []
labels = []
values = []
pos = 0
for campaign in CAMPAIGN_ORDER[1:]:
    for init_mode in INIT_ORDER:
        ts = run_for(campaign, init_mode)["transfer_summary"]
        gaps = np.asarray(ts["transfer_gap"], dtype=np.float64)
        positions.append(pos)
        labels.append(f"{CAMPAIGN_LABELS[campaign].replace('DW alpha=1/30, ', '')}\n{INIT_LABELS[init_mode]}")
        values.append(gaps)
        pos += 1
    pos += 0.7
parts = ax.violinplot(values, positions=positions, widths=0.55, showmeans=True, showextrema=True)
for body, label in zip(parts["bodies"], labels):
    body.set_facecolor("#64748b")
    body.set_alpha(0.35)
for key in ("cmeans", "cmins", "cmaxes", "cbars"):
    if key in parts:
        parts[key].set_color("#111827")
        parts[key].set_linewidth(0.8)
for x, vals in zip(positions, values):
    jitter = np.linspace(-0.12, 0.12, len(vals)) if len(vals) > 1 else np.array([0.0])
    ax.scatter(x + jitter, vals, s=10, color="#111827", alpha=0.65, zorder=3)
ax.set_xticks(positions)
ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=6)
ax.set_ylabel("Transfer gap")
ax.grid(True, axis="y", alpha=0.25, lw=0.5)
savefig(fig, "transfer_gap_summary")
plt.show()


## Executive Summary

In [ ]:

def fmt_pm(mean: float, err: float, digits: int = 4) -> str:
    return f"{mean:.{digits}g} ± {err:.{digits}g}"

lines = []
lines.append("# Executive Summary")
lines.append("")
lines.append(f"Analyzed {len(runs)} completed runs ({actual_samples} per-sample files): three campaigns, two initial states, 10 samples each, 40 cycles.")
lines.append("All run configs record `p_gain_eff=0`, `p_loss_eff=0`, `radius=6`, and `nshell=1`.")
lines.append("")
lines.append("## Final Real-Space Chern")
for _, row in final_chern_summary.iterrows():
    lines.append(
        f"- {row['campaign_label']} / {row['init_label']}: final Chern "
        f"{fmt_pm(row['final_chern_mean'], row['final_chern_sem'])}; "
        f"mean change from cycle 0 = {row['delta_chern_mean']:.4g}."
    )
lines.append("")
lines.append("## Local Chern Marker")
for _, row in local_marker_region_summary.iterrows():
    if pd.notna(row.get("local_marker_slab_mean", np.nan)):
        lines.append(
            f"- {row['campaign_label']} / {row['init_label']}: full-system mean "
            f"{fmt_pm(row['local_marker_full_mean'], row['local_marker_full_sem'])}; "
            f"DW slab mean {fmt_pm(row['local_marker_slab_mean'], row['local_marker_slab_sem'])}."
        )
    else:
        lines.append(
            f"- {row['campaign_label']} / {row['init_label']}: full-system mean "
            f"{fmt_pm(row['local_marker_full_mean'], row['local_marker_full_sem'])}."
        )
lines.append("")
lines.append("## Transfer Spectra")
if not transfer_gap_summary.empty:
    for _, row in transfer_gap_summary.iterrows():
        active_text = "all Choi trajectories active" if row["choi_active_all"] else f"{row['choi_inactive_count']} inactive Choi trajectories"
        lines.append(
            f"- {row['campaign_label']} / {row['init_label']}: transfer gap "
            f"{fmt_pm(row['transfer_gap_mean'], row['transfer_gap_sem'])}; "
            f"basis size {int(row['choi_basis_size'])}; {active_text}."
        )
else:
    lines.append("- No transfer spectra were expected for the uniform campaign.")
lines.append("")
lines.append("## Main Figures")
for stem in [
    "chern_vs_cycle_overview",
    "chern_vs_cycle_by_campaign",
    "local_chern_marker_grid",
    "local_chern_marker_init_differences",
    "transfer_exponent_spectra",
    "transfer_gap_summary",
]:
    lines.append(f"- `figures/{stem}.png` and `figures/{stem}.pdf`")

summary_text = "\n".join(lines) + "\n"
(OUT_DIR / "executive_summary.md").write_text(summary_text, encoding="utf-8")
print(summary_text)
if display is not None and Markdown is not None:
    display(Markdown(summary_text))
